In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
class DynamicGraph(nn.Module):
    def __init__(self, feature_dim=64, max_nodes=1024, max_edges_per_node=8, input_size=64, output_size=2, hidden_layers=[128, 256, 256, 128, 64], device=None):
        super().__init__()
        if device==None:
            if hasattr(torch, "xpu") and torch.xpu.is_available():
                device = torch.device("xpu")
            elif torch.cuda.is_available():
                device = torch.device("cuda")
            else:
                device = torch.device("cpu")
        self.device = torch.device(device)
        print(f"\nUsing device : {device}\n")

        self.D = feature_dim
        self.max_nodes = max_nodes
        self.max_edges = max_nodes * max_edges_per_node
        # max_edges = self.max_edges
        self.max_edges_per_node = max_edges_per_node
        self.input_size = input_size
        self.output_size = output_size
        self.hidden_layers = hidden_layers
        self.max_rank = len(hidden_layers)

        ## Node Thingys

        # Node Weights (no_of_nodes,1)
        self.node_weight = nn.Parameter(
            torch.zeros(
                max_nodes,
                feature_dim,
                device=self.device
            )
        )

        # Node Pre Activation(Input of current node) (no_of_nodes,1)
        self.node_pre_activation = nn.Parameter(
            torch.zeros(
                max_nodes,
                feature_dim,
                device=self.device
            )
        )

        # Node Bias (no_of_nodes,1)
        self.node_bias = nn.Parameter(
            torch.randn(
                max_nodes,
                feature_dim,
                device=self.device
            ) * 0.02
        )

        # Node Active/Deactive (no_of_nodes,1)
        self.register_buffer(
            "node_alive",
            torch.zeros(
                max_nodes,
                dtype=torch.bool,
                device=self.device
            )
        )

        # Node Activation Type (Relu/Sigmoid/LRelu) (no_of_nodes,1)
        self.register_buffer(
            "node_activation",
            torch.zeros(
                max_nodes,
                dtype=torch.long,
                device=self.device
            )   
        )

        # Node Out Degree (no_of_nodes,1) (Value ranges from 0 to max_edges_per_node)
        self.register_buffer(
            "out_degree",
            torch.zeros(
                max_nodes,
                dtype=torch.long,
                device=self.device
            )
        )

        # Node In Degree (no_of_nodes,1) (Value ranges from 0 to no_of_nodes * max_edges_per_node)
        self.register_buffer(
            "in_degree",
            torch.zeros(
                max_nodes,
                dtype=torch.long,
                device=self.device
            )
        )

        # Node Rank (no_of_nodes,1)
        # Used to store the rank of nodes, which is used to prevent cycles
        self.register_buffer(
            "node_rank",
            torch.zeros(
                max_nodes,
                dtype=torch.long,
                device=self.device
            )
        )

        # No of current nodes (Includes Active + Deactive)
        self.nodes_count = 0


        ## Edge Thingys

        # Edge Source[edge_source][index] = edge_destination (no_of_nodes,max_edges_per_node)
        self.register_buffer(
            "edge_src",
            -1*torch.ones(
                max_nodes,
                max_edges_per_node,
                dtype=torch.long,
                device=self.device
            )
        )

        # Edge Index[edge_destination][index] = edge_source (no_of_nodes,max_edges_per_node)
        # Used to store the index of edge stored in edge_src matrix for quick operation in back prop
        self.register_buffer(
            "edge_index",
            -1*torch.ones(
                max_nodes,
                max_edges_per_node,
                dtype=torch.long,
                device=self.device
            )
        )

        # Edge Destination[edge_destination][index] = edge_source (no_of_nodes,max_edges_per_node)
        self.register_buffer(
            "edge_dest",
            -1*torch.ones(
                max_nodes,
                max_edges_per_node,
                dtype=torch.long,
                device=self.device
            )
        )

        # Edge Weight[edge_source][edge_destination] (no_of_nodes,max_edges_per_node)
        self.edge_weight = nn.Parameter(
            torch.randn(
                max_nodes,
                max_edges_per_node,
                device=self.device
            ) * 0.02
        )

        # Edge Active/Deactive[edge_source][edge_destination] (no_of_nodes,max_edges_per_node)
        self.register_buffer(
            "edge_alive",
            torch.zeros(
                max_nodes,
                max_edges_per_node,
                dtype=torch.bool,
                device=self.device
            )   
        )

    @torch.no_grad()
    def add_node(self, activation=1):
        if self.nodes_count >= self.max_nodes:
            raise RuntimeError("Node capacity reached in Add Node"+str(self.max_nodes))
        index = self.nodes_count
        self.node_alive[index] = True
        self.node_activation[index] = activation
        self.nodes_count += 1
        return index

    @torch.no_grad()
    def remove_node(self, index):
        if index>=self.nodes_count or index<0:
            raise RuntimeError("Out of bound in Remove Node")

        self.node_alive[index] = False

    @torch.no_grad()
    def add_edge(self, source, destination):
        if source<0 or source>=self.nodes_count or destination<0 or destination>=self.nodes_count:
            raise RuntimeError("Out of bound in Add Edge 1st Thingy")
        if self.out_degree[source]>=self.max_edges_per_node or self.in_degree[destination]>=self.max_edges_per_node:
            raise RuntimeError("Out of bound in Add Edge 2nd Thingy")
        self.edge_src[source][self.out_degree[source]] = destination

        self.edge_index[destination][self.in_degree[destination]] = self.out_degree[source]
        self.edge_dest[destination][self.in_degree[destination]] = source

        self.edge_alive[source][self.out_degree[source]] = True

        self.out_degree[source] += 1
        self.in_degree[destination] += 1

    @torch.no_grad()
    def remove_edge(self, source, destination):
        if source<0 or source>=self.nodes_count or destination<0 or destination>=self.nodes_count:
            raise RuntimeError("Out of bound in Remove Edge")
        temp_index = self.edge_src[source]==destination
        self.edge_alive[source][temp_index] = False

    @torch.no_grad()
    def initialize_graph(self):
        total_layers = sum(self.hidden_layers)
        if total_layers>self.max_nodes:
            raise RuntimeError("More no of hidden thingys in Initialize Graph")
        
        current_count = 0
        temp_layers = self.hidden_layers.copy()
        print(temp_layers)
        output_layer_count = temp_layers.pop()
        print(temp_layers)
        for i in range(len(temp_layers)):
            layer_count = temp_layers[i]
            self.node_activation[current_count:current_count+layer_count] = 1
            self.node_rank[current_count:current_count+layer_count] = i+1
            current_count += layer_count
        self.node_activation[current_count:output_layer_count] = 2
        self.node_rank[current_count:output_layer_count] = len(self.hidden_layers)
        self.node_alive[:total_layers] = True
        self.nodes_count = total_layers

        source_idx = torch.randperm(
            self.hidden_layers[0],
            device=self.device
        )
        temp_idx = torch.randint(
            self.hidden_layers[0],
            self.nodes_count,
            source_idx.shape,
            device=self.device
        )
        temp_idx = torch.unique(temp_idx)
        destination_idx = temp_idx[self.in_degree[temp_idx]<self.max_edges_per_node][:self.hidden_layers[0]]
        while destination_idx.shape != source_idx.shape:
            temp_idx = torch.randint(self.hidden_layers[0],self.nodes_count,source_idx.shape)
            temp_idx = torch.unique(temp_idx)
            destination_idx = temp_idx[self.in_degree[temp_idx]<self.max_edges_per_node][:self.hidden_layers[0]]

        print(destination_idx.shape)
        print(max(destination_idx))
        print(min(destination_idx))
        print(self.nodes_count)
        
        current_count = 0
        for i in range(len(temp_layers)):
            pass
        
        


In [50]:
torch.randperm(5)

tensor([3, 2, 1, 0, 4])

In [97]:
# temp = DynamicGraph(2,10,2)
temp = DynamicGraph(2,1024)
# node0 = temp.add_node()
# node1 = temp.add_node()
# node2 = temp.add_node()

# temp.add_edge(node0,node2)
# temp.add_edge(node0,node1)
# temp.remove_edge(node0,node2)

# temp.remove_node(node1)

print(temp.node_alive)



Using device : xpu

tensor([False, False, False,  ..., False, False, False], device='xpu:0')


In [98]:
temp.initialize_graph()
temp1 = temp.node_rank
print(temp1)

[128, 256, 256, 128, 64]
[128, 256, 256, 128]


RuntimeError: indices should be either on cpu or on the same device as the indexed tensor (cpu)